# Checkpoint storage example for PyTorch DDP using Fashion MNIST Training

This example presents workflow of a checkpoint storage in the scratch-volume for a convolutional neural network (CNN) that classify images using the [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist) dataset and [PyTorch Distributed Data Parallel (DDP)](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)

The reference for the original notebook is [here](https://github.com/kubeflow/trainer/blob/master/examples/pytorch/image-classification/mnist.ipynb)

## Verification of dependencies and their versions

In [1]:
import kubeflow
print(f"Kubeflow version: {kubeflow.__version__ if hasattr(kubeflow, '__version__') else 'N/A'}")
print("All imports successful!")

Kubeflow version: 0.3.0
All imports successful!


## Define the Training Function
Create function to train CNN model using Fashion MNIST data along with their config values (number of samples, epochs, etc)

In [2]:
def train_fashion_mnist(checkpoint_path="./checkpoint.pth", save_interval=1, resume=False):
    import os
    import random
    import torch
    import torch.distributed as dist
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, DistributedSampler, Subset
    from torchvision import datasets, transforms

    # -----------------------------
    # Checkpoint saving
    # -----------------------------
    def save_checkpoint(model, optimizer, epoch, path):
        os.makedirs(path, exist_ok=True)
    
        # support both DDP and non-DDP models
        model_state = (
            model.module.state_dict()
            if hasattr(model, "module")
            else model.state_dict()
        )
    
        state = {
            "epoch": epoch,
            "model": model_state,
            "optimizer": optimizer.state_dict(),
        }
    
        latest_path = os.path.join(path, "latest.pt")
        epoch_path = os.path.join(path, f"epoch_{epoch:03d}.pt")
    
        # --- atomic save ---
        tmp_latest = latest_path + ".tmp"
        tmp_epoch = epoch_path + ".tmp"
    
        torch.save(state, tmp_latest)
        torch.save(state, tmp_epoch)
    
        # atomic rename (safe on Linux / Kubernetes volumes)
        os.replace(tmp_latest, latest_path)
        os.replace(tmp_epoch, epoch_path)
    
        print(f"Checkpoint saved at epoch {epoch} to {latest_path}")


        
    # -----------------------------
    # Define the CNN model
    # -----------------------------
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()
            self.conv1 = nn.Conv2d(1, 20, 5, 1)
            self.conv2 = nn.Conv2d(20, 50, 5, 1)
            self.fc1 = nn.Linear(4 * 4 * 50, 500)
            self.fc2 = nn.Linear(500, 10)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            x = F.max_pool2d(x, 2, 2)
            x = F.relu(self.conv2(x))
            x = F.max_pool2d(x, 2, 2)
            x = x.view(-1, 4 * 4 * 50)
            x = F.relu(self.fc1(x))
            x = self.fc2(x)
            return F.log_softmax(x, dim=1)

    # -----------------------------
    # Device and distributed setup
    # -----------------------------
    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    print(f"Using Device: {device}, Backend: {backend}")

    local_rank = int(os.getenv("LOCAL_RANK", 0))
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        dist.init_process_group(backend=backend)
        distributed = True
    else:
        distributed = False

    if distributed:
        print(
            f"WORLD_SIZE: {dist.get_world_size()}, "
            f"RANK: {dist.get_rank()}, LOCAL_RANK: {local_rank}"
        )
    else:
        print("Running in single-process mode")


    device = torch.device(f"{device}:{local_rank}")
    model = nn.parallel.DistributedDataParallel(Net().to(device))
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

    # -----------------------------
    # Training loop and checkpoint saving
    # -----------------------------
    start_epoch = 1
    checkpoint_path = "/scratch-volume/checkpoints-m-xochicale"
    save_interval=1
    resume=False
    
    # Resume from checkpoint if exists
    if resume and os.path.exists(checkpoint_path):
        map_location = {"cuda:%d" % 0: "cuda:%d" % local_rank} if torch.cuda.is_available() else "cpu"
        checkpoint = torch.load(checkpoint_path, map_location=map_location)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        start_epoch = checkpoint["epoch"] + 1
        if local_rank == 0:
            print(f"Resumed training from epoch {checkpoint['epoch']}")

    # -----------------------------
    # Load dataset
    # -----------------------------
    if local_rank == 0:
        datasets.FashionMNIST(
            "./data",
            train=True,
            download=True,
            transform=transforms.ToTensor(),
        )
    dist.barrier()
    dataset = datasets.FashionMNIST(
        "./data",
        train=True,
        download=False,
        transform=transforms.ToTensor(),
    )

    # Random subset
    num_samples = 1000
    random.seed(42)
    all_indices = list(range(len(dataset)))
    random.shuffle(all_indices)
    subset_indices = all_indices[:num_samples]
    subset_dataset = Subset(dataset, subset_indices)

    if distributed:
        sampler = DistributedSampler(subset_dataset)
        shuffle = False  # sampler handles shuffling
    else:
        sampler = None
        shuffle = True  # normal shuffling in single-process mode

    train_loader = DataLoader(
        subset_dataset,
        batch_size=100,
        sampler=sampler,
        shuffle=shuffle,
        num_workers=0,
    )

    EPOCHS = 10
    dist.barrier()


    # Determine if this is the main process
    is_main_process = not dist.is_initialized() or dist.get_rank() == 0
    # -----------------------------
    # Training loop
    # -----------------------------    
    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = F.nll_loss(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Determine rank safely
            rank = dist.get_rank() if dist.is_initialized() else 0

            # if batch_idx % 20 == 0 and dist.get_rank() == 0:
            #     print(
            #         f"Train Epoch: {epoch} [{batch_idx * len(inputs)}/{len(subset_dataset)} "
            #         f"({100.0 * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}"
            #     )

            # Only print progress every 20 batches on rank 0
            if batch_idx % 20 == 0 and rank == 0:
                print(
                    "Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                        epoch,
                        batch_idx * len(inputs),
                        len(subset_dataset),
                        100.0 * batch_idx / len(train_loader),
                        loss.item(),
                    )
                )

        # Save checkpoint at intervals or last epoch
        if dist.get_rank() == 0 and (epoch % save_interval == 0 or epoch == EPOCHS):
            save_checkpoint(model, optimizer, epoch, checkpoint_path)

    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")

    dist.destroy_process_group()

## Scale PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes.

`TrainerClient()` verifies that you have required access to the Kubernetes cluster.

Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.



In [3]:
from kubeflow.trainer import CustomTrainer, TrainerClient
client = TrainerClient()

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob.

Additionally, it might show available accelerator type and number of available resources.

In [4]:
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

Runtime(name='deepspeed-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='deepspeed', image='ghcr.io/kubeflow/trainer/deepspeed-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='mlx-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='mlx', image='ghcr.io/kubeflow/trainer/mlx-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='retfound-image-generation', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/andesterson/yukun-image-generation-runner:v0.0.8', num_nodes=1, device='gpu', device_count='1'), pretrained_model=None)
Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtim

## Run the Distributed TrainJob

Kubeflow TrainJob will train the above model on PyTorch nodes defined by `NUM_NODES` and each node with `RESOURCES_PER_NODE`.

In [5]:
from kubeflow.trainer import TrainerClient, CustomTrainer, options

# ------------------------------------------------
# Configuration of resources
# ------------------------------------------------

## Set how many PyTorch nodes you want to use for distributed training.
NUM_NODES = 1

# Set the resources for each PyTorch node.
RESOURCES_PER_NODE = {
    "cpu": "5",           # CPUs per node
    "memory": "2Gi",     # Memory in GiB per node
    "nvidia.com/gpu": 1,  # GPUs per node (the number will depend on the available resources)
}

# ------------------------------------------------
# Seeting up to mount checkpoint volume
# ------------------------------------------------
volume_name = "scratch-volume"
pvc_name = "scratch-volume"
mount_scrath_path = "/checkpoints-m-xochicale"


pod_volumes = [
    {
        "name": volume_name,
        "persistentVolumeClaim": {
            "claimName": pvc_name
        }
    }
]
volume_mounts = [
    {
        "name": volume_name,
        "mountPath": mount_scrath_path
    }
]
container_override = options.ContainerOverride(
    name="node",
    volume_mounts=volume_mounts
)

pod_spec_override = options.PodSpecOverride(
    volumes=pod_volumes,
    containers=[container_override]
)
pod_template_override = options.PodTemplateOverride(
    target_jobs=["node"],
    spec=pod_spec_override
)
pod_template_overrides = options.PodTemplateOverrides(
    pod_template_override
)

In [6]:
job_name = client.train(
    runtime=torch_runtime,
    trainer=kubeflow.trainer.CustomTrainer(
        func=train_fashion_mnist,
        num_nodes=NUM_NODES,
        resources_per_node=RESOURCES_PER_NODE,
    ),
    options=[
        pod_template_overrides
    ]
)

In [7]:
#Check job status directly
job = client.get_job(job_name)
print(f"\nJob ID name: {job_name}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")


Job ID name: g2d4b4eeaead
Job Status: Created
Creation Time: 2026-03-11 12:34:37+00:00

Job details: TrainJob(name='g2d4b4eeaead', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[], num_nodes=1, creation_timestamp=datetime.datetime(2026, 3, 11, 12, 34, 37, tzinfo=TzInfo(0)), status='Created')


In [8]:
from datetime import datetime
import time
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(client.get_job_logs(job_name, follow=True))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


Waiting for job logs...
Logs received after 0 seconds:
  bash: line 211: 1238649373.py: Permission denied
  /opt/conda/bin/python3.11: can't open file '/workspace/1238649373.py': [Errno 2] No such file or directory
  E0311 12:34:41.697000 1 site-packages/torch/distributed/elastic/multiprocessing/api.py:874] failed (exitcode: 2) local_rank: 0 (pid: 56) of binary: /opt/conda/bin/python3.11
  Traceback (most recent call last):
    File "/opt/conda/bin/torchrun", line 8, in <module>
      sys.exit(main())
               ^^^^^^
    File "/opt/conda/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 355, in wrapper
      return f(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^
    File "/opt/conda/lib/python3.11/site-packages/torch/distributed/run.py", line 892, in main
      run(args)
    File "/opt/conda/lib/python3.11/site-packages/torch/distributed/run.py", line 883, in run
      elastic_launch(
    File "/opt/conda/lib/python3.11/site-packa

# Delete the TrainJob
When TrainJob is finished, you can delete the resource.

In [9]:
client.delete_job(job_name)